In [2]:
import os

# Install the Kaggle API client
!pip install kaggle

In [3]:
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_d48c4fab5ba2cc09f8e7c4e8de4ffd4b'
os.environ['KAGGLE_USERNAME'] = 'robstark143' # Replace 'YOUR_KAGGLE_USERNAME' with your actual Kaggle username

In [4]:
# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Create the kaggle.json file with your API token and username
# Ensure these environment variables are correctly set in the previous cell
kaggle_json_content = f'{{"username":"{os.environ.get("KAGGLE_USERNAME")}","key":"{os.environ.get("KAGGLE_API_TOKEN")}"}}'
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json_content)

# Set permissions for the kaggle.json file (read/write for owner only)
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json created and permissions set.")

kaggle.json created and permissions set.


### 3. Diabetes Dataset

In [5]:
!kaggle datasets download -d mathchi/diabetes-data-set
!unzip diabetes-data-set.zip

import pandas as pd

df = pd.read_csv("diabetes.csv")
df.head()

df.columns
df.info()
df.describe()

cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]

for col in cols:
    df[col] = df[col].replace(0, df[col].mean())

df.describe()

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


import numpy as np

# Example input (change values)
new_patient = np.array([[2, 120, 70, 25, 100, 28.5, 0.5, 33]])

new_patient = scaler.transform(new_patient)
prediction = model.predict(new_patient)

if prediction[0] == 1:
    print("🔴 Diabetes Detected")
else:
    print("🟢 No Diabetes")

    import joblib

joblib.dump(model, "diabetes_model.pkl")
joblib.dump(scaler, "scaler.pkl")



Dataset URL: https://www.kaggle.com/datasets/mathchi/diabetes-data-set
License(s): CC0-1.0
  0% 0.00/8.91k [00:00<?, ?B/s]
100% 8.91k/8.91k [00:00<00:00, 15.9MB/s]
Archive:  diabetes-data-set.zip
  inflating: diabetes.csv            
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
Accura

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


['scaler.pkl']

###5. BP

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

df = pd.read_csv("diabetes.csv")
df.head()

# Create BP target variable BEFORE dropping or accessing it
df["BP_Label"] = df["BloodPressure"].apply(lambda x: 1 if x >= 80 else 0)

X = df.drop(["BloodPressure", "BP_Label"], axis=1)
y = df["BP_Label"]

print(X.columns)

print(df["BP_Label"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


with open("bp_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

with open("bp_model.pkl", "wb") as f:
    pickle.dump(model, f)

import pandas as pd
import numpy as np
import pickle

# Load scaler
with open("bp_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Create new patient in CORRECT FORMAT (8 features)
new_patient = pd.DataFrame([[
    2,      # Pregnancies
    120,    # Glucose
    20,     # SkinThickness
    85,     # Insulin
    25.6,   # BMI
    45,     # Age
    0,      # Outcome (0 = no diabetes, 1 = diabetes)
    1       # (extra feature you might have — depends on your X.columns)
]], columns=X.columns)

# Scale properly
new_patient_scaled = scaler.transform(new_patient)

# Load model
with open("bp_model.pkl", "rb") as f:
    model = pickle.load(f)

pred = model.predict(new_patient_scaled)

if pred[0] == 1:
    print("🔴 High Blood Pressure (Hypertension)")
else:
    print("🟢 Normal Blood Pressure")


def bp_check(sys, dia):
    if sys < 120 and dia < 80:
        return "Normal BP"
    elif sys < 140:
        return "Elevated BP"
    else:
        return "High BP"

def diabetes_check(glucose):
    if glucose < 100:
        return "Normal"
    elif glucose < 126:
        return "Prediabetes"
    else:
        return "Diabetes"
        # FINAL TEST OUTPUT

# Example inputs
predicted_label = "Normal"   # try: Normal / Abnormal / Benign / Malignant
sys_bp = 128
dia_bp = 84

# The 'descriptions' variable is not defined in the current notebook state.
# Skipping the execution of the following lines until 'descriptions' is defined.
# print("🧠 Scan Analysis:")
# print(descriptions[predicted_label])

print("\n🩺 BP Analysis:")
print(bp_check(sys_bp, dia_bp))


Index(['Pregnancies', 'Glucose', 'SkinThickness', 'Insulin', 'BMI',
       'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')
BP_Label
0    563
1    205
Name: count, dtype: int64
Accuracy: 0.7272727272727273

Report:
               precision    recall  f1-score   support

           0       0.80      0.86      0.83       118
           1       0.38      0.28      0.32        36

    accuracy                           0.73       154
   macro avg       0.59      0.57      0.58       154
weighted avg       0.70      0.73      0.71       154

🟢 Normal Blood Pressure

🩺 BP Analysis:
Elevated BP


### 8.liver-patient

In [8]:
!kaggle datasets download -d uciml/indian-liver-patient-records
!unzip indian-liver-patient-records.zip

import os
print(os.listdir())

import pandas as pd

df = pd.read_csv("indian_liver_patient.csv")
print(df.head())
print(df.shape)
print(df.columns)

df.info()

print(df.isnull().sum())

df.rename(columns={"Dataset": "label"}, inplace=True)

X = df.drop(columns=["label"])
y = df["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.head())

# Convert Male -> 1, Female -> 0
X_train["Gender"] = X_train["Gender"].map({"Male": 1, "Female": 0})
X_test["Gender"] = X_test["Gender"].map({"Male": 1, "Female": 0})

from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
with open("liver_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
with open("liver_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))

import pickle

with open("liver_model.pkl", "wb") as f:
    pickle.dump(model, f)

import numpy as np
import pickle

# Example new patient (use your own values)
# Format: [Age, Gender(1=Male,0=Female), Total_Bilirubin, Direct_Bilirubin,
#          Alkaline_Phosphotase, Alamine_Aminotransferase,
#          Aspartate_Aminotransferase, Total_Proteins, Albumin, A/G_Ratio]

new_patient = np.array([[60, 1, 0.5, 0.2, 180, 40, 120, 6.8, 3.8, 1.2]])

with open("liver_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("liver_model.pkl", "rb") as f:
    model = pickle.load(f)

new_scaled = scaler.transform(new_patient)
pred = model.predict(new_scaled)

print("Prediction:", "Liver Disease" if pred[0]==1 else "Healthy")

from google.colab import files

files.download("liver_model.pkl")

files.download("liver_scaler.pkl")

import pickle

with open("liver_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("liver_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)



Dataset URL: https://www.kaggle.com/datasets/uciml/indian-liver-patient-records
License(s): CC0-1.0
  0% 0.00/7.68k [00:00<?, ?B/s]
100% 7.68k/7.68k [00:00<00:00, 3.99MB/s]
Archive:  indian-liver-patient-records.zip
  inflating: indian_liver_patient.csv  
['.config', 'indian_liver_patient.csv', 'mitbih_train.csv', 'scaler.pkl', 'ptbdb_abnormal.csv', 'diabetes_model.pkl', 'diabetes.csv', 'heartbeat.zip', 'diabetes-data-set.zip', 'indian-liver-patient-records.zip', 'mitbih_test.csv', 'ptbdb_normal.csv', 'sample_data']
   Age  Gender  Total_Bilirubin  Direct_Bilirubin  Alkaline_Phosphotase  \
0   65  Female              0.7               0.1                   187   
1   62    Male             10.9               5.5                   699   
2   62    Male              7.3               4.1                   490   
3   58    Male              1.0               0.4                   182   
4   72    Male              3.9               2.0                   195   

   Alamine_Aminotransferase

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10.Kidney Disease

In [9]:
!kaggle datasets download -d mansoordaku/ckdisease
!unzip ckdisease.zip

import pandas as pd

df = pd.read_csv("kidney_disease.csv")
df.head()
df.info()
df.describe()

df["classification"].value_counts()

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
df[df.select_dtypes(include=['float64','int64']).columns] = imputer.fit_transform(
    df.select_dtypes(include=['float64','int64'])
)

df = pd.get_dummies(df, drop_first=True)

print(df.columns)

print(df.columns.tolist())
df.columns = df.columns.str.strip()
print(df.columns.tolist())

import pandas as pd
import numpy as np

df = pd.read_csv("/content/kidney_disease.csv")

df['classification'] = df['classification'].str.strip().str.lower()

df['classification'].value_counts()

df['classification'] = df['classification'].map({
    'ckd': 1,
    'notckd': 0
})

df = df.dropna(subset=['classification'])

df['classification'].isna().sum()

X = df.drop('classification', axis=1)
y = df['classification']

X = X.replace('?', np.nan)
X = X.apply(pd.to_numeric, errors='coerce')

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

df['classification'] = df['classification'].map({
    'ckd': 1,
    'notckd': 0
})

import joblib

joblib.dump(model, "ckd_model.pkl")
joblib.dump(scaler, "ckd_scaler.pkl")
joblib.dump(imputer, "ckd_imputer.pkl")



Dataset URL: https://www.kaggle.com/datasets/mansoordaku/ckdisease
License(s): unknown
  0% 0.00/9.51k [00:00<?, ?B/s]
100% 9.51k/9.51k [00:00<00:00, 28.7MB/s]
Archive:  ckdisease.zip
  inflating: kidney_disease.csv      
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float6

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['rbc' 'pc' 'pcc' 'ba' 'htn' 'dm' 'cad' 'appet' 'pe' 'ane']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


['ckd_imputer.pkl']